# Molab 01: GNN Baselines (Official Multi-GNN + TD-GNN) — completing Gate G1

**Goal:** Finish the official-model work the whole project's main claim depends on: does the published Multi-GNN result (and TD-GNN, our own view-restricted model) actually depend on look-ahead and on the benchmark's post-simulation tail? `docs/PHASE3_PLAN.md` calls this "the biggest open question": the official Multi-GNN had never been run at all before the GPU campaign this notebook completes.

**Covers:** NB1 (official Multi-GIN+EU, HI-Small, 3 seeds + replays), NB2 (official Multi-PNA+EU HI-Small 2 seeds, Multi-GIN+EU LI-Small 2 seeds + replays, FraudGT audit), and NB3's remaining P1 work (the full horizon grid, 3 ablations, LI-Small attacks, GPU service time — NB3's P0 is already 100% complete and verified, so none of it is repeated here).

**A real change from the plan's own assumptions, worth noting up front:** every GPU run so far used a 10 GB RTX 3080, where the official Multi-PNA setting (`batch_size=8192`) does not fit and needed a documented deviation (`docs/DECISIONS.md` D-022/D-023, batch 512 or 1024). Molab's 96 GB card has **13x** that memory, so this notebook runs Multi-PNA at the frozen **official batch size, unmodified** — the deviation this project has been carrying simply does not apply here.

**Expected wall time on the stated hardware:** 3 to 10 hours, depending heavily on how much of NB1's 3 full 100-epoch seeds remain after reuse (one is already finished and verified in `main`'s `results/`, so it is skipped automatically). **Requires a GPU.** The next cell stops cleanly with a clear message if none is attached, after finishing setup so nothing is repeated once you attach one and re-run.

## What makes this notebook self-contained
Every cell below is idempotent and safe to re-run. On a fresh Molab session it, in order: (1) clones `aml-stream`'s
`main` branch from GitHub (public, no token needed — set `GITHUB_TOKEN` as an environment variable/secret only if the
repository has since been made private); (2) downloads the AMLworld CSVs it needs directly from Kaggle's public
download endpoint (no `kaggle.json` needed, with a hash check against `docs/SPEC.md`'s verified values, because one
download endpoint has silently truncated a file before); (3) installs the exact pinned dependencies; (4) runs the
test suite as a gate — **a failing test stops the notebook before it spends any compute**; (5) reuses whatever
earlier, *verified* output already exists in the cloned repository (so PC-8's and PC-18's already-finished jobs are
never repeated); (6) runs its job list under a wall-clock budget that always stops cleanly before the 12 h session
limit, in a way that is safe to interrupt and resume (`Run all` again after a restart continues from the last
checkpoint, it does not start over); (7) writes a human-readable analysis with tables and plots; (8) zips the output
folder so it survives the session ending.

**Nothing here needs you to upload or attach anything by hand.**

In [ ]:
# ============================= Settings: the only cell you would normally change =============================
import asyncio, os, sys, time   # asyncio is used by later cells via asyncio.to_thread(...), so it is imported once, here
NOTEBOOK = 'NB1'
REPO_URL = os.environ.get('AML_REPO_URL', 'https://github.com/Shahoud867/aml-stream.git')
REPO_BRANCH = os.environ.get('AML_REPO_BRANCH', 'main')   # pin a commit instead by setting this to a SHA
BUDGET_HOURS = float(os.environ.get('AML_BUDGET_HOURS', 11.5))   # stops cleanly well inside Molab's 12 h cap
TIERS = ('P0', 'P1')
RUN_TESTS = True     # a failing test stops the notebook before it spends any compute
INSTALL = True

SESSION_START = time.time()
def hours_left():
    return BUDGET_HOURS - (time.time() - SESSION_START) / 3600

# --- GPU ---
import subprocess
_GPU = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
HAS_GPU = _GPU.returncode == 0 and bool(_GPU.stdout.strip())
print("GPU:", _GPU.stdout.strip() or "none found")


In [ ]:
# ============================= Where things live =============================
# A plain, dependency-free layout that works on any Jupyter kernel (Molab, a local machine, Colab, ...): everything
# lives under the user's home directory, so no platform-specific mount point is assumed.
from pathlib import Path

BASE = Path(os.environ.get('AML_WORKDIR', Path.home() / 'aml_molab'))
BASE.mkdir(parents=True, exist_ok=True)
CODE = BASE / 'aml-stream'               # the cloned repository
INPUT = BASE / 'data'                    # raw AMLworld files
CACHE = BASE / 'cache'                   # prepared (parquet) data, rebuilt once and reused across cells
WORKING = BASE / f'out_{NOTEBOOK.lower()}'
WORKING.mkdir(parents=True, exist_ok=True)
for p in (INPUT, CACHE):
    p.mkdir(parents=True, exist_ok=True)
print('workdir', BASE)
print('  code   ->', CODE)
print('  data   ->', INPUT)
print('  cache  ->', CACHE)
print('  output ->', WORKING)

In [ ]:
# ============================= Get the code =============================
# Clones (or fast-forwards) the repository. The expected code hash is computed from whatever we actually cloned,
# never hard-coded, so this notebook can never go stale the way a fixed-hash code package would.
#
# git clone/fetch run via asyncio.to_thread: a plain blocking subprocess.run() call here (or anywhere else long-
# running in this notebook) blocks this kernel's own async event loop for its whole duration. Confirmed live,
# pairing this exact notebook: that is enough for something outside this project's code (consistent with an
# unresponsiveness watchdog) to interrupt the cell outright. to_thread moves the blocking wait off the event
# loop's own thread, so the loop stays free to answer whatever is checking on it, the whole time.
import subprocess

def run(cmd, **kw):
    r = subprocess.run(cmd, text=True, capture_output=True, **kw)
    if r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stdout}\n{r.stderr}")
    return r.stdout

token = os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN')
clone_url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
if not (CODE / '.git').exists():
    print('cloning', REPO_URL, '...')
    await asyncio.to_thread(run, ['git', 'clone', '--branch', REPO_BRANCH, clone_url, str(CODE)])
else:
    print('updating existing clone ...')
    await asyncio.to_thread(run, ['git', '-C', str(CODE), 'fetch', 'origin', REPO_BRANCH])
    run(['git', '-C', str(CODE), 'checkout', REPO_BRANCH])        # local, fast: no network wait worth threading
    run(['git', '-C', str(CODE), 'reset', '--hard', f'origin/{REPO_BRANCH}'])

sys.path.insert(0, str(CODE / 'src'))
from amlh.runner import code_hash
CODE_HASH = code_hash(CODE)
print('HEAD', run(['git', '-C', str(CODE), 'rev-parse', 'HEAD']).strip())
print('code hash', CODE_HASH)

In [ ]:
# ============================= Install dependencies =============================
# Off the event loop via asyncio.to_thread -- see the note in the "Get the code" cell above.
if INSTALL:
    await asyncio.to_thread(run, [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements-kaggle.txt')])
    print('requirements-kaggle.txt installed')

    # torch + torch-geometric + the official Multi-GNN and FraudGT repos at their pinned commits.
    from amlh.session import setup_environment
    if HAS_GPU:
        info = await asyncio.to_thread(setup_environment, 'NB2', official_root=BASE / 'official')   # 'NB2' installs everything NB1 needs too
        print(info)
    else:
        print("No GPU: skipping the official Multi-GNN/FraudGT/torch install. This notebook cannot do its GPU work")
        print("without a GPU; see the next cell, which stops here rather than pretending to proceed.")


In [ ]:
# ============================= GPU gate =============================
# This notebook trains official Multi-GNN models and TD-GNN, which need a GPU to finish in any reasonable time (the
# project's own measurements: roughly 1 to 1.3 GPU-hours per 100-epoch Multi-GIN seed on a much smaller card than
# Molab's; on CPU the same run would be impractical). Rather than silently running a misleading, barely-started
# fraction of the work, this stops here with a clear message if no GPU is attached — "fail loudly, not silently"
# is this project's own stated rule for exactly this situation (see gpu_session/README.md).
assert HAS_GPU, (
    "This notebook needs a GPU (Molab: attach the RTX PRO 6000 Blackwell and re-run). "
    "Everything up to here (code, data, environment, tests) is done and verified, so re-running after "
    "attaching a GPU will not repeat any of it."
)

In [ ]:
# ============================= Raw data =============================
# Downloaded directly from Kaggle's public endpoint (no account needed) and checked against the row counts and
# SHA-256 hashes in docs/SPEC.md -- the same check every notebook in this project runs, because one Kaggle download
# endpoint has silently truncated a file before while still reporting success.
# Off the event loop via asyncio.to_thread -- see the note in the "Get the code" cell above. HI-Medium and
# LI-Medium are several GB each; without this, a download that size is exactly the kind of long blocking call
# that gets this cell interrupted outright (confirmed live: this happened before to_thread was added here).
from amlh.fetch import ensure_raw
from amlh.session import verify_raw

DATASETS = ['HI-Small', 'LI-Small']
print('fetching', DATASETS)
await asyncio.to_thread(ensure_raw, DATASETS, INPUT)
for d in DATASETS:
    print(d, verify_raw(d, INPUT))

In [ ]:
# ============================= Tests: the gate =============================
# Off the event loop via asyncio.to_thread -- see the note in the "Get the code" cell above. The full suite
# takes several minutes; run synchronously that is easily long enough to get this cell interrupted outright.
from amlh.session import run_preflight_tests
if RUN_TESTS:
    await asyncio.to_thread(run_preflight_tests, CODE)
    print('all tests passed')

In [ ]:
# ============================= Reuse prior verified work =============================
# The freshly cloned repository's own results/ folder holds every job this project has finished so far, in exactly
# the job-folder layout the runner understands. Pointing it there for free: a job already verified elsewhere is
# never repeated, and a job that was interrupted partway (with a usable checkpoint) resumes instead of restarting.
PRIOR_RESULTS = [CODE / 'results']
print('will reuse any verified job found under', [str(p) for p in PRIOR_RESULTS])

In [ ]:
# ============================= Run the job queue off the event loop =============================
# amlh.session.run_notebook's own job loop is a plain synchronous `for` loop: gbt_sweep (LightGBM) and the
# polling loop around an isolated mgnn_train/tdgnn_train subprocess both run entirely in-process, with no
# `await` anywhere inside them. A single call that runs for a long time therefore blocks this kernel's own
# async event loop for that whole time. Observed live, pairing this exact notebook with the marimo-pair skill:
# a long call like this gets interrupted outright, confirmed from the runner's own ledger to be `stop_requested`
# becoming true (not this project's own time-budget check: remaining() was still far above what the next step
# needed) -- consistent with something outside this project's code deciding the kernel looked unresponsive and
# signalling it. The same thing happened to the clone, install, data-fetch and test cells above, which is why
# they are wrapped in asyncio.to_thread too.
#
# An earlier version of this fix chopped run_notebook into many short, repeated slices with a sleep in between,
# to force the event loop to get a turn. That introduced its own bug: run_notebook() restores from every prior
# result and re-verifies the raw files on every call, and repeating that every few minutes (instead of once)
# consumed each slice's entire budget before a new job could ever start -- confirmed live, DONE froze for dozens
# of consecutive slices while the budget drained. asyncio.to_thread is the more direct fix for the actual
# problem: it moves run_notebook's blocking work onto its own OS thread, so the event loop stays free for the
# whole call, in one piece, not an artificially chopped-up one -- no slicing, no redundant re-verification.
from amlh.session import run_notebook


async def run_notebook_sliced(notebook, budget_hours, **kw):
    return await asyncio.to_thread(run_notebook, notebook, budget_hours=budget_hours, **kw)

## Run NB1, NB2, and NB3's remaining work

Each of the three keeps using `amlh.session.run_notebook`, the exact tested orchestrator the project's Kaggle/Colab
notebooks use: per-epoch checkpointing, resumable replays, a wall-clock budget that stops cleanly, and jobs that
fail loudly (not silently) on a real error. The budget below is split across the three, biased toward NB1 since it
carries the heaviest remaining work (up to 3 full 100-epoch seeds); each stops early and hands its leftover time to
the next if it finishes sooner than its share.

In [ ]:
from amlh.session import print_summary, final_checklist

# A fixed split of this session's remaining budget. NB1 (Multi-GIN, up to 3 full seeds) gets the largest share;
# NB2 (Multi-PNA + Multi-GIN LI-Small + FraudGT) the next; NB3 (only P1 remains) the smallest, since its P0 is done.
SHARES = {'NB1': 0.45, 'NB2': 0.40, 'NB3': 0.15}
summaries = {}
for nb, share in SHARES.items():
    left = hours_left()
    if left <= 0.1:
        print(f'{nb}: no time left in this session, skipping (re-run this cell after extending BUDGET_HOURS or in a new session)')
        continue
    budget = min(left, max(left * share / sum(s for n, s in SHARES.items() if n not in summaries), 0.25))
    print(f'=== {nb}: {budget:.2f} h budget, {left:.2f} h left in the session ===')
    nb_work = WORKING / nb.lower()
    summaries[nb] = await run_notebook_sliced(nb, budget_hours=budget, tiers=TIERS, input_root=INPUT, working=nb_work,
                                               restore_roots=PRIOR_RESULTS, cache_root=CACHE, run_name='aml_run',
                                               official_root=BASE / 'official')
    print_summary(nb_work)
    final_checklist(nb, TIERS, nb_work, platform='molab')


## Gate G1: does the official model show the look-ahead / tail effect?

Reuses `amlh.merge`'s own functions directly — the same code the final merge step (notebook 03) and
`scripts/08_analyze.py` use — so this table is computed the identical, tested way, not re-derived by hand.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from amlh.merge import collect, official_audit, leakage_check
from amlh.jobs_gbt import get_stream, Paths

Paths.raw = {d: INPUT for d in DATASETS}
Paths.cache = CACHE

roots = [WORKING] + PRIOR_RESULTS
c = collect(roots)
print(f"{len(c.jobs)} verified jobs found across this session's output and the reused prior work; {len(c.problems)} problems")
for p in c.problems[:10]:
    print(' problem:', p)

rows = []
for ds in ('HI-Small',):   # LI-Small is sealed: this notebook must not compute or display its test metrics
    s, _ = get_stream({'dataset': ds, 'protocol': 'released'})
    audit = official_audit(c, s)
    for jid, a in sorted(audit.items()):
        if 'problem' in a:
            continue
        r = a['row']
        rows.append({'job': jid, 'dataset': ds, 'H': a['H'], 'official_style_f1': a['official_style_f1'],
                     'published_reference': a['published_reference'], 'core_f1': r['test_core']['f1'],
                     'tail_f1': r.get('test_tail', {}).get('f1'), 'core_pr_auc': r['test_core']['pr_auc']})
    leak = leakage_check(c, s)
    for jid, v in leak.items():
        print(f"leakage check {jid}: core PR-AUC {v['core_pr_auc']:.4f} vs prevalence {v['prevalence']:.5f} "
              f"-> {'PASS' if v['pass'] else 'FAIL -- investigate before trusting any positive-horizon result'}")

if rows:
    print(f"{'job':45s} {'H':>8s} {'official F1':>12s} {'published':>10s} {'core F1':>8s} {'tail F1':>8s}")
    for r in rows:
        pub = f"{r['published_reference']:.4f}" if r['published_reference'] is not None else '-'
        tail = f"{r['tail_f1']:.4f}" if r['tail_f1'] is not None else '-'
        print(f"{r['job']:45s} {str(r['H']):>8s} {r['official_style_f1']:>12.4f} {pub:>10s} {r['core_f1']:>8.4f} {tail:>8s}")
    (WORKING / 'gate_g1_audit.json').write_text(json.dumps(rows, indent=1))
    labels = [r['job'][:30] for r in rows]
    x = np.arange(len(rows))
    fig, ax = plt.subplots(figsize=(max(6, len(rows) * 0.6), 4))
    ax.bar(x - 0.2, [r['official_style_f1'] for r in rows], 0.4, label='reproduced (official-style F1)')
    ax.bar(x + 0.2, [r['published_reference'] or 0 for r in rows], 0.4, label='published reference')
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=60, ha='right'); ax.set_ylabel('minority-class F1')
    ax.set_title('Gate G1: reproduced vs published, by job and cadence'); ax.legend(); plt.tight_layout()
    plt.savefig(WORKING / 'gate_g1.png', dpi=130); plt.show()
else:
    print('No official-model jobs verified yet in this session or in the reused prior results -- Gate G1 remains open.')


## TD-GNN: core PR-AUC by decision horizon (RQ1)

The same computation used earlier in this project's own analysis, now reading whatever this session produced plus
whatever NB3's already-complete P0 output (reused above) contributes — should reproduce it closely on HI-Small,
and extend it with the P1 horizons / ablations this session ran.

In [ ]:
import re
from collections import defaultdict
import statistics as st

td_rows = []
for root in roots:
    for jd in Path(root).rglob('nb3-tdgnn-*'):
        if not jd.is_dir() or not (jd / 'metrics.json').exists():
            continue
        m = re.match(r'nb3-tdgnn-(HI-Small|LI-Small)-H(\w+)-s(\d+)$', jd.name)
        if not m:
            continue
        ds, h, seed = m.groups()
        met = json.loads((jd / 'metrics.json').read_text())
        td_rows.append((ds, h, int(seed), met))

by = defaultdict(list)
for ds, h, seed, met in td_rows:
    by[(ds, h)].append(met)

HORDER = ['0', '1', '60', '600', '3600', '21600', '86400', '259200', 'inf']
print(f"{'H':>8s} {'n':>2s} {'val PR-AUC':>11s} {'test PR-AUC':>12s} {'core PR-AUC':>12s} {'tail PR-AUC':>12s}")
curve = []
for h in HORDER:
    if ('HI-Small', h) not in by:
        continue
    ms = by[('HI-Small', h)]
    def avg(path):
        vals = [path(m) for m in ms if path(m) is not None]
        return st.mean(vals) if vals else None
    v = avg(lambda m: m.get('val_pr_auc'))
    t = avg(lambda m: m.get('test_results', {}).get('test', {}).get('pr_auc'))
    co = avg(lambda m: m.get('test_results', {}).get('test_core', {}).get('pr_auc'))
    ta = avg(lambda m: m.get('test_results', {}).get('test_tail', {}).get('pr_auc'))
    print(f"{h:>8s} {len(ms):>2d} {v or 0:>11.4f} {t or 0:>12.4f} {co or 0:>12.4f} {ta or 0:>12.4f}")
    curve.append((h, co))
if curve:
    fig, ax = plt.subplots(figsize=(6, 4))
    xs = range(len(curve))
    ax.plot(xs, [c[1] or 0 for c in curve], 'o-')
    ax.set_xticks(xs); ax.set_xticklabels([c[0] for c in curve], rotation=45)
    ax.set_xlabel('decision horizon H'); ax.set_ylabel('core PR-AUC'); ax.set_title('TD-GNN, HI-Small: does waiting help?')
    plt.tight_layout(); plt.savefig(WORKING / 'td_horizon_curve.png', dpi=130); plt.show()


In [ ]:
# ============================= Package the output =============================
# Molab sessions are not guaranteed to persist, so the output is zipped here (and the notebook's own cell outputs,
# saved with the .ipynb itself, carry the printed tables below even if the zip is never downloaded).
import shutil
archive = shutil.make_archive(str(BASE / f'gnn_baselines_{NOTEBOOK.lower()}'), 'zip', WORKING)
print('wrote', archive, f'({Path(archive).stat().st_size / 1e6:.1f} MB)')
print()
print('Download this file before your Molab session ends. If you plan to run another of the three notebooks next,')
print('the most useful thing to carry over is the whole', WORKING, 'folder (or this zip, unpacked next to the new')
print("notebook's own working directory) -- the next notebook's merge step will pick up anything placed under")
print("its PRIOR_RESULTS / EXTRA_ROOTS list.")

## What to do next

- If you still have session time: increase `BUDGET_HOURS` and `Run all` again. Nothing above repeats finished work.
- Otherwise: download the zip this notebook wrote, or push `results/` back to a branch of this repository, and run
  **Notebook 03** next — point its `EXTRA_ROOTS` at this notebook's output folder (or its unzipped contents) to
  fold these results into the final merge and benchmark report.